# Lab 3 · EASE

The runnable companion of the lab page, `docs/labs/03-ease.md` (also on the docs site). The page explains each step,
asks the questions and holds the hints and solutions; this notebook has the code.

- Run the setup cell first, every time the kernel starts (after a crash: **Restart**, then setup again).
- Cells that fit models take seconds to minutes on MovieLens. Run them where you train: on the rented box (VS Code's
  Remote - SSH extension opens this notebook there), or on the laptop if you allow training there.
- Cells that only read stored results (the baseline, `lab.compare`, `lab.segments`) are fine anywhere, once the box's
  results are fetched (`./scripts/fetch_lab_results.sh`).
- Save the notebook **without outputs** before you commit (the **...** menu → Clear All Outputs).

In [ ]:
from recbench import lab

lab.setup()
DATASET = "movielens-25m"
METHOD = "ease"

## Your starting point

The lab baseline on every dataset (stored results; nothing is trained here).

In [ ]:
lab.baseline(METHOD)

## Level 1: reproduce and read

### 1.1 Reproduce the baseline's best setting

Fit the baseline's best setting on the validation fold, in this notebook. The NDCG@10 must equal the baseline's validation score.

In [ ]:
best = lab.baseline(METHOD, DATASET)["best_settings"]
data, split = lab.load(DATASET)
model = lab.fit(METHOD, data, **best)
result = lab.evaluate(model, data, split)
print(result)
print("the baseline's best trial:", lab.baseline(METHOD, DATASET)["validation"])

### 1.2 Read the code

The questions are on the lab page; answer them with the code open next to this notebook.

### 1.3 Look inside

In [ ]:
import numpy as np

item = int(data.item_pop.argmax())
row = model.weights[model.position[item]]  # how much having this item predicts each kept item
order = np.argsort(row)
print("weights from", data.item_text[item] or data.item_ids[item])
for j in list(order[::-1][:5]) + list(order[:5]):
    other = int(model.kept[j])
    print(f"{row[j]:+.4f}  {data.item_text[other] or data.item_ids[other]}")
print(model.fit_info)  # how many items the cap kept, and the share of interactions they cover

**Exercise.** EASE's weights can be negative. What share of all weights is negative, and are the most negative ones larger or smaller in size than the most positive ones?

<details><summary>Hint</summary>

`model.weights` is a dense numpy array: `(model.weights < 0).mean()`, `model.weights.min()`, `model.weights.max()`.

</details>

<details><summary>Solution</summary>

```python
w = model.weights
print(f"negative: {(w < 0).mean():.1%}; most negative {w.min():.4f}; most positive {w.max():.4f}")
```
Typically about half the weights are negative, but small; the positive ones carry the strongest links. Negative weights encode "having i makes j less likely", information SLIM throws away (lab 4).

</details>

In [ ]:
# Your code here

## Level 2: understand each setting

For each sweep: **write your prediction in the cell below it first**, then run the sweep. Every other setting stays at the baseline's best. The lab page explains what to look for.

In [ ]:
table = lab.sweep(METHOD, DATASET, "ease_lambda", [1, 10, 100, 1000, 10000], start="best")
lab.plot_sweep(table)
table

**Your prediction for `ease_lambda`, and what you saw:** ...

## Level 3: data tricks

Experiments run on the rented box over all five datasets, for example:

```bash
./scripts/run_lab_box.sh ease:no-time-knobs
```

Then fetch the results (`./scripts/fetch_lab_results.sh <host>`) and compare here. `no-time-knobs` is ready to run; `keep-alltime-popular`
needs a small code change first (the lab page has the hint and the folded solution). After writing it, run its test:
`pytest -q tests/test_ease_variant.py`.

In [ ]:
lab.compare(METHOD, f"{METHOD}:no-time-knobs")

In [ ]:
lab.compare(METHOD, f"{METHOD}:keep-alltime-popular")

## Level 4: one change from the literature

Write the variant (the lab page explains the idea, with hints and the folded solution), test it, uncomment
`edlae` in `labs/03-ease/experiments.yaml`, run it on the box, fetch, then compare and look at the segments.

In [ ]:
lab.compare(METHOD, f"{METHOD}:edlae")

In [ ]:
parts = lab.segments(METHOD, f"{METHOD}:edlae", DATASET)
lab.plot_segments(parts, "activity")

In [ ]:
parts["items"]

## Record your results

Fill in the table at the end of the lab page, then rebuild the scoreboard: `python -m recbench.lab scoreboard --docs`.